# bxai: Bayesian Feature Selection and Attribution Suite

This notebook demonstrates each of the core examples found in the `bxai` README.md. 

Ensure that you have selected the registered **Python (bxai)** kernel to run this notebook.

### Load Data

In [30]:
from sklearn.datasets import load_breast_cancer

# Load Breast Cancer dataset
data = load_breast_cancer(as_frame=True)
X, y = data.data, data.target
print(f"{len(X)} Rows and {len(X.columns)} Columns")

569 Rows and 30 Columns


## 1. Global Non-Linear Selection: `BayesianBorutaSHAP`

`BayesianBorutaSHAP` performs tree-based feature selection using SHAP values, replacing frequentist p-values with Bayesian credible intervals. It supports both discrete (Beta-Binomial) and continuous (Normal-Inverse-Gamma) modes.

In [ ]:
import lightgbm as lgb
from bxai.selection import BayesianBorutaSHAP

# Fit Bayesian BorutaSHAP
clf = lgb.LGBMClassifier(random_state=42, verbose=-1)
selector = BayesianBorutaSHAP(model=clf, 
                                mode="discrete", 
                                max_iter=100,
                                random_state=42)
selector.fit(X, y)

print("Confirmed Features:")
selector.confirmed_

Confirmed Features:


['mean texture',
 'mean smoothness',
 'mean concave points',
 'area error',
 'compactness error',
 'worst radius',
 'worst texture',
 'worst perimeter',
 'worst area',
 'worst smoothness',
 'worst concavity',
 'worst concave points']

## 2. Global Non-Linear Selection: `BayesianPermutation`

`BayesianPermutation` is a model-agnostic feature selection tool that tracks importance using paired validation loss drops updated via the Student-t continuous engine.

In [56]:
from bxai.selection import BayesianPermutation
from sklearn.ensemble import RandomForestClassifier

# Fit classifier
clf_perm = RandomForestClassifier(random_state=42, verbose=0).fit(X.values, y)

# Permutation feature selection with parallel jobs (n_jobs=2)
selector_perm = BayesianPermutation(
    model=clf_perm,
    scoring="accuracy",
    n_repeats=100,
    n_jobs=2,
    random_state=42,
)
selector_perm.fit(X, y)

print("Confirmed Features:", selector_perm.confirmed_)
selected_df = selector_perm.summary()
selected_df[selected_df['status'] == 'Confirmed'].sort_values(by ='mean')

Confirmed Features: ['mean texture', 'mean concave points', 'radius error', 'area error', 'compactness error', 'worst texture', 'worst area', 'worst smoothness', 'worst concave points']


,feature,status,mean,hdi_lower,hdi_upper,nu,alpha,beta
24,worst smoothness,Confirmed,0.000562,0.000238,0.000887,100.0001,50.0001,0.000134
13,area error,Confirmed,0.001160,0.000834,0.001486,100.0001,50.0001,0.000135
23,worst area,Confirmed,0.001213,0.000860,0.001565,100.0001,50.0001,0.000158
27,worst concave points,Confirmed,0.001318,0.000960,0.001676,100.0001,50.0001,0.000163
15,compactness error,Confirmed,0.001336,0.001018,0.001653,100.0001,50.0001,0.000128
10,radius error,Confirmed,0.001388,0.001074,0.001703,100.0001,50.0001,0.000126
21,worst texture,Confirmed,0.001564,0.001263,0.001865,100.0001,50.0001,0.000115
1,mean texture,Confirmed,0.002285,0.001926,0.002643,100.0001,50.0001,0.000163
7,mean concave points,Confirmed,0.002654,0.002254,0.003054,100.0001,50.0001,0.000203


## 3. Local Interpretability: `BayLIME`

`BayLIME` generates stable, prior-informed local explanations wrapping standard or custom perturbations in a Bayesian linear regression.

In [54]:
from bxai.explanation import BayLIME
import numpy as np

# Instantiate BayLIME
explainer = BayLIME(
    training_data=X,
    feature_names=list(X.columns),
    backend = 'mcmc'
)

# Fit the base model before explaining (since BorutaSHAP clones the model internally)
clf.fit(X, y)

# Explain a single instance
explanation = explainer.explain_instance(
    instance=X.iloc[0].values,
    predict_fn=clf.predict_proba
)

explanation.as_dataframe()

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (2 chains in 2 jobs)
NUTS: [intercept, coefs, sigma_global]


EOFError: 

## 4. Parametric Selection: `ShrinkagePIP`

`ShrinkagePIP` implements high-dimensional generalized linear models (GLMs) using Horseshoe and Lasso regularizing priors, tracking Posterior Inclusion Probabilities (PIP) to select features.

In [52]:
from bxai.parametric import ShrinkagePIP

# Horseshoe prior — uses kappa-based PIP by default (pip_method='auto')
# PIP = P(κ_j < 0.5 | data), where κ_j = 1/(1 + λ_j² τ²)
selector_hs = ShrinkagePIP(
    model_type="logistic",
    prior="horseshoe",
    kappa_threshold=0.5,   # κ < 0.5 → local scale dominates → signal
    pip_threshold=0.80,
    n_samples=500,
    random_state=42,
)
selector_hs.fit(X, y)

print("\nSelected features (Horseshoe):", selector_hs.confirmed_)
selector_hs.summary()


Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (2 chains in 2 jobs)
NUTS: [intercept, tau, lambdas, beta]
Sampling 2 chains for 1_000 tune and 500 draw iterations (2_000 + 1_000 draws total) took 1 seconds.
There were 1000 divergences after tuning. Increase `target_accept` or reparameterize.
We recommend running at least 4 chains for robust computation of convergence diagnostics
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details



Selected features (Horseshoe): ['symmetry error']


,feature,pip,pip_method,selected,mean,std,hdi_lower,hdi_upper,interval_type,kappa_mean
0,mean radius,0.0,kappa,False,-0.041258,0.704435,-0.745694,0.663177,hdi,0.781568
1,mean texture,0.0,kappa,False,0.767192,0.056998,0.710193,0.824190,hdi,0.653300
2,mean perimeter,0.0,kappa,False,0.174614,0.664510,-0.489897,0.839124,hdi,0.652024
3,mean area,0.5,kappa,False,-0.008173,0.224454,-0.232627,0.216282,hdi,0.530976
4,mean smoothness,0.5,kappa,False,0.369280,0.250249,0.119031,0.619531,hdi,0.462126
5,mean compactness,0.5,kappa,False,-0.205772,0.401670,-0.607442,0.195898,hdi,0.679417
6,mean concavity,0.0,kappa,False,0.171078,0.524909,-0.353832,0.695988,hdi,0.783377
7,mean concave points,0.0,kappa,False,-0.334074,0.607242,-0.941317,0.273168,hdi,0.786244
8,mean symmetry,0.0,kappa,False,-0.772494,0.026022,-0.798516,-0.746471,hdi,0.847167
9,mean fractal dimension,0.0,kappa,False,-0.011117,0.804753,-0.815870,0.793635,hdi,0.877508


In [48]:

# Lasso prior — uses auto-scaled |β| threshold (epsilon = 0.1)
selector_lasso = ShrinkagePIP(
    model_type="logistic",
    prior="lasso",
    pip_threshold=0.80,
    n_samples=500,
    random_state=42,
)
selector_lasso.fit(X, y)
print(f"\nEffective epsilon: {selector_lasso.epsilon_:.4f}")
print("\nSelected features (LASSO):", selector_lasso.confirmed_)
selector_lasso.summary()

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (2 chains in 2 jobs)
NUTS: [intercept, b, beta]
Sampling 2 chains for 1_000 tune and 500 draw iterations (2_000 + 1_000 draws total) took 1 seconds.
There were 1000 divergences after tuning. Increase `target_accept` or reparameterize.
We recommend running at least 4 chains for robust computation of convergence diagnostics
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details



Effective epsilon: 0.1000

Selected features (LASSO): ['mean radius', 'mean texture', 'mean perimeter', 'mean area', 'mean smoothness', 'mean compactness', 'mean concavity', 'mean concave points', 'mean symmetry', 'mean fractal dimension', 'radius error', 'texture error', 'perimeter error', 'area error', 'smoothness error', 'concavity error', 'concave points error', 'symmetry error', 'fractal dimension error', 'worst radius', 'worst texture', 'worst perimeter', 'worst area', 'worst smoothness', 'worst concave points', 'worst symmetry', 'worst fractal dimension']


,feature,pip,pip_method,selected,mean,std,hdi_lower,hdi_upper,interval_type,epsilon
0,mean radius,1.0,threshold,True,-0.041527,0.704704,-0.746232,0.663177,hdi,0.1
1,mean texture,1.0,threshold,True,0.766706,0.057484,0.709220,0.824190,hdi,0.1
2,mean perimeter,1.0,threshold,True,0.173416,0.665708,-0.492292,0.839124,hdi,0.1
3,mean area,1.0,threshold,True,-0.008146,0.224481,-0.232627,0.216336,hdi,0.1
4,mean smoothness,1.0,threshold,True,0.370122,0.251091,0.119031,0.621214,hdi,0.1
5,mean compactness,1.0,threshold,True,-0.205665,0.401563,-0.607229,0.195898,hdi,0.1
6,mean concavity,1.0,threshold,True,0.172776,0.526607,-0.353832,0.699384,hdi,0.1
7,mean concave points,1.0,threshold,True,-0.335734,0.608902,-0.944637,0.273168,hdi,0.1
8,mean symmetry,1.0,threshold,True,-0.772368,0.026147,-0.798516,-0.746219,hdi,0.1
9,mean fractal dimension,1.0,threshold,True,-0.010264,0.803900,-0.814165,0.793635,hdi,0.1


## 5. Native Bayesian Importance: `BARTImportance`

`BARTImportance` tracks Variable Inclusion Frequencies (VIF) from native Bayesian Additive Regression Trees (BART) models to estimate feature importance.

In [51]:
from bxai.parametric import BARTImportance

# Classification example (Probit BART)
bart_clf = BARTImportance(model_type="classification", 
                            n_trees=500, 
                            n_samples=200, 
                            tune=100, 
                            chains=1, 
                            random_state=42)
bart_clf.fit(X, y)
print("Classification Selected features:", bart_clf.confirmed_)

Sequential sampling (1 chains in 1 job)
PGBART: [mu]
Sampling 1 chain for 100 tune and 200 draw iterations (100 + 200 draws total) took 36 seconds.
Only one chain was sampled, this makes it impossible to run some convergence checks


Classification Selected features: []
